# Week 26, Day 5: Review V: information, dynamics and stochastic processes

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w026/d5.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Problem 1: how many bits does the count carry?

Four stimuli each evoke Poisson counts with means 2, 4, 6 and 8 over 50 trials each. Write `mutual_info(s, r)`, the plug-in mutual information in bits from the joint histogram. Compute `I_plugin`, then subtract the mean over 200 shuffles of the stimulus labels (local rng seed 21) to get `I_corrected`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
ri = np.random.default_rng(20)
stim = np.repeat(np.arange(4), 50)
counts = ri.poisson(np.array([2, 4, 6, 8])[stim])

def mutual_info(s, r):
    J = np.zeros((s.max() + 1, r.max() + 1)); np.add.at(J, (s, r), 1); J /= J.sum()
    ps, pr = J.sum(1), J.sum(0)
    nz = J > 0
    return ...

I_plugin = mutual_info(stim, counts)
rs = np.random.default_rng(21)
bias = np.mean([mutual_info(rs.permutation(stim), counts) for _ in range(200)])
I_corrected = ...
print(f"plug-in {I_plugin:.3f} bits, shuffle bias {bias:.3f}, corrected {I_corrected:.3f} bits (maximum 2)")

In [ ]:
import numpy as np
def test_plugin():
    "I_plugin is the plug-in mutual information in bits"
    J = np.zeros((4, counts.max() + 1)); np.add.at(J, (stim, counts), 1); J /= J.sum()
    ps, pr = J.sum(1), J.sum(0)
    nz = J > 0
    assert abs(I_plugin - np.sum(J[nz] * np.log2(J[nz] / np.outer(ps, pr)[nz]))) < 1e-9

def test_corrected():
    "the shuffle-corrected value is below the plug-in value and between 0 and 2 bits"
    assert 0 < I_corrected < I_plugin < 2

test_plugin()
test_corrected()
print("All checks passed")

### Exercise: Problem 2: the rhythm of an E-I circuit

Linearised excitatory and inhibitory populations obey ẋ = Ax with A = [[(wEE − 1)/τE, −wEI/τE], [wIE/τI, −(1 + wII)/τI]] (τE = 10 ms, τI = 5 ms, wEE = 1.5, wEI = 2.5, wIE = 3, wII = 0.5). Compute the trace `tr` and determinant `det`, the predicted frequency `f_pred` in Hz, then simulate a kick (RK4, dt = 0.05 ms, 200 ms) and measure `f_meas` from the mean spacing of upward zero crossings of x_E.

In [ ]:
tE, tI, wEE, wEI, wIE, wII = 10.0, 5.0, 1.5, 2.5, 3.0, 0.5
A = np.array([[(wEE - 1) / tE, -wEI / tE], [wIE / tI, -(1 + wII) / tI]])
tr, det = np.trace(A), np.linalg.det(A)
f_pred = ...                                      # Hz, from the imaginary part sqrt(det - tr^2/4) per ms
x, dt, xs = np.array([1.0, 0.0]), 0.05, []
for _ in range(4000):
    k1 = A @ x; k2 = A @ (x + dt / 2 * k1); k3 = A @ (x + dt / 2 * k2); k4 = A @ (x + dt * k3)
    x = x + dt / 6 * (k1 + 2 * k2 + 2 * k3 + k4); xs.append(x[0])
xs = np.array(xs)
up = np.where((xs[:-1] < 0) & (xs[1:] >= 0))[0]
f_meas = ...                                      # 1000 / (mean spacing of upward crossings in ms)
print(f"trace {tr:.3f}, det {det:.3f}: stable spiral;  predicted {f_pred:.1f} Hz, measured {f_meas:.1f} Hz")

In [ ]:
import numpy as np
def test_kind():
    "the fixed point is a stable spiral: trace below 0 and trace squared below 4 det"
    assert tr < 0 and tr ** 2 < 4 * det

def test_freq():
    "the frequency measured from the simulation is within 3% of the eigenvalue prediction"
    assert abs(f_meas - f_pred) / f_pred < 0.03

test_kind()
test_freq()
print("All checks passed")

### Exercise: Problem 3: a membrane's time constant from its noise

A membrane's voltage fluctuations follow an OU process with τ = 10 ms and σ = 1, sampled every 0.1 ms for 100 s (generated exactly below). Fit x[k+1] = a x[k] + noise by least squares (`a_hat`), recover τ from a = e^(−Δt/τ) (`tau_hat`), and σ from the residual variance, which equals σ²τ(1 − a²)/2 (`sigma_hat`).

In [ ]:
ro = np.random.default_rng(24)
tau_true, sig_true, dt = 10.0, 1.0, 0.1
a_true = np.exp(-dt / tau_true); sd_step = sig_true * np.sqrt(tau_true / 2 * (1 - a_true ** 2))
n = 1_000_000
noise = ro.normal(size=n) * sd_step
v = np.empty(n); v[0] = ro.normal() * sig_true * np.sqrt(tau_true / 2)
for k in range(1, n):
    v[k] = a_true * v[k - 1] + noise[k]
a_hat = ...
tau_hat = ...
resid = v[1:] - a_hat * v[:-1]
sigma_hat = ...
print(f"a = {a_hat:.5f};  tau = {tau_hat:.2f} ms;  sigma = {sigma_hat:.3f}")

In [ ]:
import numpy as np
def test_a():
    "a_hat is the least-squares slope of x[k+1] on x[k]"
    assert abs(a_hat - np.sum(v[:-1] * v[1:]) / np.sum(v[:-1] ** 2)) < 1e-12

def test_tau():
    "tau_hat is within 5% of 10 ms and sigma_hat within 5% of 1"
    assert abs(tau_hat - 10) / 10 < 0.05 and abs(sigma_hat - 1) < 0.05

test_a()
test_tau()
print("All checks passed")